# ASSISTments: predicción con Random Forest

Este cuaderno de exposición **lee un único archivo ya preparado**: `assistments_rf_dataset.csv.gz`. No une archivos originales ni construye otra tabla. Entrena un solo algoritmo, Random Forest, y muestra las métricas y gráficos en Colab.

Cada fila es el resultado de un problema de evaluación de fin de unidad. `score = 1` indica éxito y `score = 0`, resultado no exitoso. Los identificadores sirven para rastreo y se excluyen del modelo. El CSV preparado contiene una muestra reproducible de 3.000 estudiantes (51.042 filas y 722 variables predictoras) para que la demostración sea manejable.

## 1. Cargar el CSV preparado

Descarga `assistments_rf_dataset.csv.gz` y súbelo cuando Colab abra el selector de archivos. Esta es la única entrada que usa el cuaderno; no hace uniones ni recortes de los datos.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from google.colab import files


SEED = 42
N_ESTIMATORS = 600  # Valor usado por el artículo.
print('Selecciona el CSV único preparado: assistments_rf_dataset.csv.gz')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Sube exactamente un archivo: assistments_rf_dataset.csv.gz')
uploaded_name = next(iter(uploaded))
if not uploaded_name.endswith('.csv.gz'):
    raise ValueError('El archivo debe ser un CSV comprimido con extensión .csv.gz')
DATASET_CSV = Path('/content') / uploaded_name
df = pd.read_csv(DATASET_CSV, low_memory=False)
required = {'score', 'student_id', 'source_split', 'unit_test_assignment_log_id', 'problem_id'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Faltan columnas requeridas en el CSV: {sorted(missing)}')

print(f'CSV cargado: {DATASET_CSV.name}')
print(f'{len(df):,} filas | {df.student_id.nunique():,} estudiantes | {df.shape[1]:,} columnas')
display(df['source_split'].value_counts(dropna=False).rename_axis('partición').to_frame('filas'))
display(df['score'].value_counts(dropna=False).sort_index().rename_axis('score').to_frame('filas'))

plt.figure(figsize=(6, 4))
sns.countplot(data=df[df['score'].notna()], x='score', color='#4C78A8')
plt.title('Distribución del resultado real')
plt.xlabel('score: 0 = no exitoso, 1 = exitoso')
plt.ylabel('Cantidad de problemas')
plt.tight_layout()
plt.show()


## 2. Entrenar y evaluar Random Forest

Si `evaluation` contiene etiquetas reales, se usa esa partición. En caso contrario, se reserva el 20 % de los estudiantes de `train` para validación, evitando que una misma persona aparezca en ambos conjuntos. El modelo predice `score`: 0 = resultado no exitoso en una pregunta y 1 = exitoso. Los identificadores y `source_split` se excluyen de las variables predictoras.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

id_columns = {
    'score', 'student_id', 'source_split',
    'unit_test_assignment_log_id', 'problem_id'
}
feature_columns = [column for column in df.columns if column not in id_columns]
if not feature_columns:
    raise ValueError('El CSV no contiene variables predictoras.')

# Solo se convierten a números las columnas que ya vienen preparadas en el CSV.
X = df[feature_columns].apply(pd.to_numeric, errors='coerce').fillna(-1).astype('float32')
y = pd.to_numeric(df['score'], errors='coerce')
labeled = y.notna() & y.isin([0, 1])
df = df.loc[labeled].reset_index(drop=True)
X = X.loc[labeled].reset_index(drop=True)
y = y.loc[labeled].astype('int8').reset_index(drop=True)
if df.empty:
    raise ValueError('No hay etiquetas score válidas (0 o 1) en el CSV.')

train_mask = df['source_split'].eq('train')
eval_mask = df['source_split'].eq('evaluation')
if eval_mask.any() and train_mask.any():
    train_idx = np.flatnonzero(train_mask.to_numpy())
    test_idx = np.flatnonzero(eval_mask.to_numpy())
    evaluation_name = 'Partición evaluation del CSV'
else:
    train_labeled_idx = np.flatnonzero(train_mask.to_numpy())
    groups = df.loc[train_labeled_idx, 'student_id'].astype('string').to_numpy()
    if len(np.unique(groups)) < 2:
        raise ValueError('Se necesitan al menos dos estudiantes etiquetados para validar.')
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    relative_train, relative_test = next(splitter.split(
        X.iloc[train_labeled_idx], y.iloc[train_labeled_idx], groups=groups
    ))
    train_idx = train_labeled_idx[relative_train]
    test_idx = train_labeled_idx[relative_test]
    evaluation_name = 'Validación interna por estudiante (80/20)'

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx].to_numpy()
y_test = y.iloc[test_idx].to_numpy()
if len(np.unique(y_train)) < 2:
    raise ValueError('El entrenamiento quedó con una sola clase. Aumenta MAX_STUDENTS en preparación.')
if len(test_idx) == 0:
    raise ValueError('La partición de evaluación está vacía.')

rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=19,
    min_samples_split=10,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
probabilities = rf.predict_proba(X_test)
class_index = {int(label): index for index, label in enumerate(rf.classes_)}
y_prob_success = probabilities[:, class_index[1]]
y_prob_non_success = probabilities[:, class_index[0]]
y_non_success = (y_test == 0).astype('int8')

metrics = {
    'Evaluación': evaluation_name,
    'Filas entrenamiento': len(train_idx),
    'Filas evaluación': len(test_idx),
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision (score=1)': precision_score(y_test, y_pred, pos_label=1, zero_division=0),
    'Recall (score=1)': recall_score(y_test, y_pred, pos_label=1, zero_division=0),
    'F1 (score=1)': f1_score(y_test, y_pred, pos_label=1, zero_division=0),
    'Precision macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
    'Recall macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
    'F1 macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
    'Recall score=0': recall_score(y_test, y_pred, pos_label=0, zero_division=0),
    'ROC AUC (score=0)': (
        roc_auc_score(y_non_success, y_prob_non_success)
        if len(np.unique(y_non_success)) == 2 else np.nan
    )
}
print('Algoritmo aplicado: Random Forest')
display(pd.DataFrame([metrics]).round(4))
print('Hiperparámetros aplicados:')
parameters_used = {
    'n_estimators': rf.n_estimators,
    'max_depth': rf.max_depth,
    'min_samples_split': rf.min_samples_split,
    'min_samples_leaf': rf.min_samples_leaf,
    'max_features': rf.max_features,
    'bootstrap': rf.bootstrap,
    'random_state': rf.random_state,
    'n_jobs': rf.n_jobs
}
display(pd.Series(parameters_used, name='valor').to_frame())


## 3. Gráficos de evaluación

El gráfico de barras compara accuracy, métricas macro y recall de `score=0`. La matriz de confusión resume aciertos y errores; la curva ROC y su AUC muestran la discriminación de resultados no exitosos al variar el umbral. La importancia de variables indica aporte predictivo, no causalidad.

Cada fila representa una pregunta, así que `score=0` no diagnostica por sí solo el riesgo global de un estudiante. Random Forest no usa curva de pérdida como las redes neuronales.


In [ ]:
performance = {
    'Accuracy': metrics['Accuracy'],
    'Precision macro': metrics['Precision macro'],
    'Recall macro': metrics['Recall macro'],
    'F1 macro': metrics['F1 macro'],
    'Recall score=0': metrics['Recall score=0']
}
fig, ax = plt.subplots(figsize=(9, 4.5))
metric_names = list(performance.keys())
metric_values = list(performance.values())
bars = ax.bar(metric_names, metric_values, color='#4C78A8')
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Valor (0 a 1)')
ax.set_title('Resumen de rendimiento de Random Forest')
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['No exitoso (0)', 'Exitoso (1)']
).plot(ax=axes[0], colorbar=False, cmap='Blues', values_format='d')
axes[0].set_title('Matriz de confusión')

if len(np.unique(y_non_success)) == 2:
    fpr, tpr, _ = roc_curve(y_non_success, y_prob_non_success)
    auc_risk = metrics['ROC AUC (score=0)']
    axes[1].plot(fpr, tpr, color='#F58518', label=f'AUC = {auc_risk:.3f}')
    axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1.02)
    axes[1].set_xlabel('Tasa de falsos positivos')
    axes[1].set_ylabel('Tasa de verdaderos positivos')
    axes[1].legend(loc='lower right')
    axes[1].set_title('Curva ROC: clase no exitosa (score=0)')
else:
    axes[1].text(0.5, 0.5, 'ROC no definida: la evaluación tiene una sola clase.',
                 ha='center', va='center', wrap=True)
    axes[1].set_axis_off()
plt.tight_layout()
plt.show()

importance = pd.Series(rf.feature_importances_, index=feature_columns).sort_values(ascending=False)
top_features = importance.head(15).sort_values()
plt.figure(figsize=(9, 6))
top_features.plot(kind='barh', color='#54A24B')
plt.title('15 variables más importantes en Random Forest')
plt.xlabel('Importancia relativa')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()


## Cómo leer las métricas y explicar el gráfico

- **Accuracy:** proporción total de predicciones correctas.
- **Precision, recall y F1 (`score=1`):** miden el reconocimiento de resultados exitosos.
- **Métricas macro:** calculan el promedio entre `score=0` y `score=1`; dan el mismo peso a ambas clases.
- **Recall (`score=0`):** de los resultados no exitosos reales, qué proporción detectó el modelo.
- **ROC AUC (`score=0`):** capacidad de ordenar resultados no exitosos por encima de exitosos al variar el umbral.

En la matriz de confusión, cada fila es la clase real y cada columna la clase predicha. Los valores de la diagonal son aciertos; los valores fuera de ella son errores. La curva de rendimiento compara métricas entre 0 y 1. Este resultado es una réplica didáctica reducida, no una reproducción exacta: el artículo usa el conjunto completo y genera 32 componentes BERT para la descripción de habilidad; este CSV codifica esa descripción como categorías y no incorpora las tres banderas de imagen, ecuación y video. Por eso las métricas pueden diferir aunque se mantenga el Random Forest y sus hiperparámetros principales.


## 4. Etapa predictiva: registros sin etiqueta

La primera celda demuestra la inferencia con filas reservadas de `evaluation`: al modelo solo se le entregan las variables predictoras, sin `score`. Es una simulación reproducible; para predecir datos realmente nuevos, ejecuta después la celda opcional y sube un CSV que ya tenga las mismas 722 variables preparadas. Los identificadores son opcionales. El resultado incluye la clase predicha y las probabilidades estimadas.

In [ ]:
# Demostración automática de inferencia en 10 filas reservadas.
# score no forma parte de demo_X ni se entrega a predict().
demo_n = min(10, len(test_idx))
demo_positions = test_idx[:demo_n]
demo_X = X.iloc[demo_positions].copy()
demo_predictions = rf.predict(demo_X)
demo_probabilities = rf.predict_proba(demo_X)

demo_id_columns = [c for c in [
    'student_id', 'unit_test_assignment_log_id', 'problem_id'
] if c in df.columns]
demo_output = df.iloc[demo_positions][demo_id_columns].reset_index(drop=True)
demo_output['score_predicho'] = demo_predictions.astype('int8')
demo_output['probabilidad_score_0'] = demo_probabilities[:, class_index[0]]
demo_output['probabilidad_score_1'] = demo_probabilities[:, class_index[1]]
demo_output['probabilidad_score_0'] = demo_output['probabilidad_score_0'].round(4)
demo_output['probabilidad_score_1'] = demo_output['probabilidad_score_1'].round(4)
print('Demostración: predicciones sobre filas reservadas; la etiqueta real se ocultó al predictor.')
display(demo_output)
demo_output_path = '/content/predicciones_demo_assistments.csv'
demo_output.to_csv(demo_output_path, index=False)
print(f'Archivo de salida guardado en: {demo_output_path}')

### Predicción sobre un CSV realmente nuevo (opcional)

Por defecto, esta celda no abre otro selector para que **Ejecutar todas** termine con el CSV único original. Cuando tengas datos nuevos, activa `EJECUTAR_PREDICCION_REAL`, ejecuta solo esta celda y sube el archivo sin `score`. Debe contener las mismas variables predictoras ya transformadas que el CSV de referencia; los registros crudos requieren pasar antes por el mismo proceso de preparación.

In [ ]:
EJECUTAR_PREDICCION_REAL = False  # Cambia a True solo cuando tengas un CSV nuevo sin score.

if not EJECUTAR_PREDICCION_REAL:
    print('Modo demostración. Para datos realmente nuevos, cambia EJECUTAR_PREDICCION_REAL a True y ejecuta esta celda.')
else:
    nuevos_archivos = files.upload()
    if len(nuevos_archivos) != 1:
        raise ValueError('Sube exactamente un archivo CSV o CSV comprimido con .gz.')
    nombre_nuevo = next(iter(nuevos_archivos))
    if not (nombre_nuevo.lower().endswith('.csv') or nombre_nuevo.lower().endswith('.csv.gz')):
        raise ValueError('El archivo debe terminar en .csv o .csv.gz.')
    ruta_nueva = Path('/content') / Path(nombre_nuevo).name
    ruta_nueva.write_bytes(nuevos_archivos[nombre_nuevo])
    nuevos_df = pd.read_csv(ruta_nueva, low_memory=False)
    if nuevos_df.empty:
        raise ValueError('El archivo nuevo no contiene filas.')
    columnas_faltantes = sorted(set(feature_columns) - set(nuevos_df.columns))
    if columnas_faltantes:
        raise ValueError(f'Faltan {len(columnas_faltantes)} variables predictoras. Ejemplos: {columnas_faltantes[:10]}')
    if 'score' in nuevos_df.columns and nuevos_df['score'].notna().any():
        print('Aviso: el archivo contiene score; se ignorará y no se usará para predecir.')
    X_nuevo = nuevos_df[feature_columns].apply(pd.to_numeric, errors='coerce').fillna(-1).astype('float32')
    pred_nuevas = rf.predict(X_nuevo)
    prob_nuevas = rf.predict_proba(X_nuevo)
    columnas_id_nuevas = [c for c in ['student_id', 'unit_test_assignment_log_id', 'problem_id'] if c in nuevos_df.columns]
    salida_nuevos = nuevos_df[columnas_id_nuevas].copy().reset_index(drop=True)
    salida_nuevos['score_predicho'] = pred_nuevas.astype('int8')
    salida_nuevos['probabilidad_score_0'] = prob_nuevas[:, class_index[0]].round(4)
    salida_nuevos['probabilidad_score_1'] = prob_nuevas[:, class_index[1]].round(4)
    print(f'Predicciones generadas: {len(salida_nuevos):,} filas')
    display(salida_nuevos.head(20))
    archivo_salida = '/content/predicciones_nuevos_datos.csv'
    salida_nuevos.to_csv(archivo_salida, index=False)
    print(f'Archivo completo guardado en: {archivo_salida}')